# ⚡ EV Infrastructure Optimization: Final Results

This notebook visualizes the outcomes of the high-performance optimization model and the subsequent **Grid Feasibility Analysis**.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
import os

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

## 1. Grid Feasibility Dashboard

We transition from 100% Demand Satisfaction (Road) to actual **Grid Realities**.

In [ ]:
grid_results_path = "../data/processed/grid_feasibility_results.parquet"
gdf_grid = gpd.read_parquet(grid_results_path)

status_counts = gdf_grid['grid_status'].value_counts()
status_chargers = gdf_grid.groupby('grid_status')['final_n'].sum()

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Status Count
sns.barplot(x=status_counts.index, y=status_counts.values, ax=ax[0], palette=['salmon', 'orange', 'seagreen'])
ax[0].set_title('Station Feasibility Count')
ax[0].set_ylabel('Number of Stations')

# Total Chargers
sns.barplot(x=status_chargers.index, y=status_chargers.values, ax=ax[1], palette=['salmon', 'orange', 'seagreen'])
ax[1].set_title('Total Chargers by Grid Status')
ax[1].set_ylabel('Total Chargers (Including Existing)')

plt.tight_layout()
plt.show()

## 2. Infrastructure Comparison

Breakdown by station type (Existing, Gas, Greenfield).

In [ ]:
type_summary = gdf_grid.groupby(['type', 'grid_status']).size().unstack(fill_value=0)
type_summary.plot(kind='bar', stacked=True, color=['orange', 'seagreen', 'salmon'])
plt.title('Grid Feasibility by Station Type')
plt.ylabel('Number of Stations')
plt.legend(title='Grid Status')
plt.show()

## 3. Interactive Feasibility Map

- **Green**: 🟢 Feasible (Substation < 10km & Has Capacity)
- **Orange**: 🟠 Grid Bottleneck (Substation < 10km BUT Overloaded)
- **Red**: 🔴 No Grid Access (No Substation within 10km)

In [ ]:
m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='CartoDB dark_matter')

status_colors = {
    'Feasible': 'green',
    'Grid Bottleneck': 'orange',
    'No Grid Access (>10km)': 'red'
}

marker_cluster = MarkerCluster(name="Infrastructure Feasibility").add_to(m)

gdf_vis = gdf_grid.to_crs(epsg=4326)

for _, row in gdf_vis.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color=status_colors.get(row['grid_status'], 'gray'),
        fill=True,
        fill_opacity=0.7,
        popup=f"Site {row['site_id']}<br>Type: {row['type']}<br>Status: {row['grid_status']}<br>Required New KW: {row['new_kw']:.0f}"
    ).add_to(marker_cluster)

folium.LayerControl().add_to(m)
m